In [1]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Load your dataset
data = pd.read_csv("final_director_starring_music.csv")

# Preprocess the data if needed (handle missing values, encode categorical variables)

In [2]:
data

,MovieID,Title,Genres,Director,Starring,DBpediaURI,UserID,Rating,Music Composer,DBpedia URI
0,1,Toy Story (1995),Animation|Children's|Comedy,NaN,NaN,NaN,1,5,NaN,NaN
1,1,Toy Story (1995),Animation|Children's|Comedy,NaN,NaN,NaN,6,4,NaN,NaN
2,1,Toy Story (1995),Animation|Children's|Comedy,NaN,NaN,NaN,8,4,NaN,NaN
3,1,Toy Story (1995),Animation|Children's|Comedy,NaN,NaN,NaN,9,5,NaN,NaN
4,1,Toy Story (1995),Animation|Children's|Comedy,NaN,NaN,NaN,10,5,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1000204,3952,"Contender, The (2000)",Drama|Thriller,NaN,NaN,NaN,5812,4,NaN,NaN
1000205,3952,"Contender, The (2000)",Drama|Thriller,NaN,NaN,NaN,5831,3,NaN,NaN
1000206,3952,"Contender, The (2000)",Drama|Thriller,NaN,NaN,NaN,5837,4,NaN,NaN
1000207,3952,"Contender, The (2000)",Drama|Thriller,NaN,NaN,NaN,5927,1,NaN,NaN


In [3]:
data.head()

,MovieID,Title,Genres,Director,Starring,DBpediaURI,UserID,Rating,Music Composer,DBpedia URI
0,1,Toy Story (1995),Animation|Children's|Comedy,NaN,NaN,NaN,1,5,NaN,NaN
1,1,Toy Story (1995),Animation|Children's|Comedy,NaN,NaN,NaN,6,4,NaN,NaN
2,1,Toy Story (1995),Animation|Children's|Comedy,NaN,NaN,NaN,8,4,NaN,NaN
3,1,Toy Story (1995),Animation|Children's|Comedy,NaN,NaN,NaN,9,5,NaN,NaN
4,1,Toy Story (1995),Animation|Children's|Comedy,NaN,NaN,NaN,10,5,NaN,NaN


In [4]:
# Step 1: Create User-Movie Matrix
# Create a pivot table
user_movie_matrix = data.pivot_table(index='UserID', columns='MovieID', values='Rating')

# Fill NaN values with a specified value (e.g., 0)
user_movie_matrix.fillna(0, inplace=True)

# Display the pivot table
user_movie_matrix

MovieID,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
UserID,,,,,,,,,,,,,,,,,,,,,
1,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6036,0.0,0.0,0.0,2.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6037,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6038,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [37]:
from sklearn.neighbors import NearestNeighbors

# Step 2: KNN on User-Movie Matrix
# Initialize KNN model
k = 2000  # Number of neighbors (including the user itself)
knn = NearestNeighbors(n_neighbors=k, metric='cosine')

# Fit the model
knn.fit(user_movie_matrix)

NearestNeighbors(metric='cosine', n_neighbors=2000)

To form 15 groups, each with 10 group members, based on similarity between users using KNN with the user-movie matrix, you can follow these steps:

Create User-Movie Matrix: Construct a matrix where rows represent users, columns represent movies, and each cell contains the rating given by the user to the corresponding movie.

Normalize Data: Normalize the user-movie matrix to ensure that ratings are on the same scale.

KNN on User-Movie Matrix: Apply KNN on the user-movie matrix to find the most similar users for each user.

Form Groups: For each user, select the top 9 most similar users and form groups consisting of the user and these similar users.

Save Groups: Save the formed groups into a suitable format.

In [38]:
import numpy as np

# Step 3: Form Groups
# Form groups for each user
groups = []
num_groups = 15  # Limit to 15 groups
group_count = 0

for i in range(len(user_movie_matrix)):
    # Check if we have formed the desired number of groups
    if group_count >= num_groups:
        break

    # Find k nearest neighbors for the current user
    _, neighbor_indices = knn.kneighbors([user_movie_matrix.iloc[i]])

    # Exclude the user itself from the list of neighbors
    neighbor_indices = neighbor_indices[0][1:]

    # Shuffle the nearest neighbor indices with a fixed random state for reproducibility
    np.random.seed(42)  # Set a fixed random state
    np.random.shuffle(neighbor_indices)

    # Form a group including the user and the top 9 most similar users
    group = [i]  # Add the user itself
    group.extend(neighbor_indices[:9])  # Add the top 9 most similar users
    
    # Add the group to the list of groups
    groups.append(group)
    group_count += 1

We use cosine similarity as the distance metric in KNN to find the most similar users.
For each user, we form a group consisting of the user itself and the top 9 most similar users based on the KNN results.
Finally, we save each formed group into separate CSV files named 'group_1.csv', 'group_2.csv', etc.

In [39]:
# Step 4: Save Groups
# Save the formed groups into a suitable format (e.g., CSV)
#for i, group in enumerate(groups):
#    group_df = user_movie_matrix.iloc[group]
#    group_df.to_csv(f"group_{i+1}.csv", index=False)

Group Modeling

In [40]:
# Step 5: User Profile Aggregation (Average Strategy)
group_profiles = []
for group in groups:
    # Select the users in the current group
    group_data = user_movie_matrix.iloc[group]

    # Calculate the mean rating for each movie across all users in the group
    group_profile = group_data.mean(axis=0)  # Calculate mean rating for each movie

    # Store the aggregated user profile for the current group
    group_profiles.append(group_profile)

# Convert the list of aggregated profiles into a DataFrame
group_profiles_df = pd.DataFrame(group_profiles)

# Display the aggregated user profiles
group_profiles_df

MovieID,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
0,3.0,0.4,0.0,0.0,0.3,1.2,0.0,0.0,0.0,1.4,...,0.0,0.0,0.0,0.2,0.0,1.1,0.5,0.0,0.0,0.5
1,1.5,0.3,0.4,0.0,0.2,1.4,0.6,0.2,0.0,1.2,...,0.0,0.0,0.0,0.4,0.4,0.0,0.0,0.0,0.0,0.4
2,2.8,0.6,0.2,0.3,0.0,0.8,0.3,0.3,0.0,1.4,...,0.4,0.0,0.0,0.0,0.2,0.6,0.9,0.0,0.0,0.2
3,0.9,0.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.6,0.0,0.0,0.0,0.0
4,2.1,0.4,0.0,0.3,0.0,1.6,0.1,0.0,0.0,0.3,...,0.0,0.0,0.0,0.1,0.0,0.4,0.9,0.0,0.0,0.6
5,2.2,0.5,0.8,0.6,0.7,0.2,0.7,0.0,0.0,0.6,...,0.0,0.0,0.0,0.0,0.5,1.0,0.0,0.0,0.0,0.0
6,0.8,0.4,0.0,0.0,0.0,1.5,0.4,0.3,0.3,1.2,...,0.0,0.0,0.0,0.0,0.0,0.3,0.0,0.0,0.0,0.0
7,3.6,1.1,1.1,0.9,0.9,1.9,1.0,0.0,0.5,1.6,...,0.0,0.0,0.0,0.0,0.0,2.2,1.5,0.0,0.0,0.8
8,1.2,0.2,0.3,0.1,0.2,1.3,0.3,0.0,0.3,0.5,...,0.4,0.0,0.0,0.2,0.3,1.1,0.9,0.0,0.0,0.4
9,3.0,2.0,1.2,0.0,0.0,1.7,1.2,0.4,0.2,1.4,...,0.0,0.0,0.0,0.0,0.0,0.4,0.0,0.0,0.0,0.0


GRS Evaluatuon Stage

In [41]:
from surprise import Dataset, Reader, SVD
import pandas as pd

# Assuming 'group_profiles_df' is the DataFrame containing the aggregated user profiles
# Unpivot the DataFrame to convert it into the format expected by Surprise library
unpivoted_df = pd.melt(group_profiles_df, var_name='MovieID', value_name='Rating', ignore_index=False).reset_index()
unpivoted_df = unpivoted_df.rename(columns={'index': 'UserID'})

# Define the rating scale (already normalized)
reader = Reader(rating_scale=(0, 1))

# Load the dataset from the unpivoted DataFrame
data = Dataset.load_from_df(unpivoted_df[['UserID', 'MovieID', 'Rating']], reader)

# Use SVD algorithm
algo = SVD()

# Train the model
trainset = data.build_full_trainset()
algo.fit(trainset)

# Evaluate the model (for demonstration purpose)
predictions = algo.test(trainset.build_testset())

# Print RMSE (Root Mean Squared Error)
from surprise import accuracy
rmse = accuracy.rmse(predictions)
print("RMSE:", rmse)

RMSE: 0.3431
RMSE: 0.34306034967792337


In [14]:
unpivoted_df

,UserID,MovieID,Rating
0,0,1,4.9
1,1,1,1.7
2,2,1,1.8
3,3,1,0.0
4,4,1,3.3
...,...,...,...
55585,10,3952,0.4
55586,11,3952,0.0
55587,12,3952,0.3
55588,13,3952,0.0


In [15]:
from surprise import SVD
from surprise import Dataset, Reader
from surprise.model_selection import cross_validate
import pandas as pd

# Unpivot the group_profiles DataFrame
unpivoted_group_profiles = group_profiles_df.unstack().reset_index()
unpivoted_group_profiles.columns = ['MovieID', 'Group', 'Rating']

# Create a Reader object specifying the rating scale
reader = Reader(rating_scale=(0, 1))  # Assuming ratings are normalized between 0 and 1

# Load the data into the Surprise Dataset
data = Dataset.load_from_df(unpivoted_group_profiles[['Group', 'MovieID', 'Rating']], reader)

# Initialize the SVD algorithm
algo = SVD()

# Perform 5-fold cross-validation
results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

# Print the evaluation results
print("Mean RMSE:", results['test_rmse'].mean())
print("Mean MAE:", results['test_mae'].mean())

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.4300  0.4425  0.4380  0.4447  0.4432  0.4397  0.0053  
MAE (testset)     0.2054  0.2065  0.2071  0.2086  0.2073  0.2070  0.0011  
Fit time          0.93    0.85    1.14    1.11    1.05    1.02    0.11    
Test time         0.09    0.11    0.12    0.13    0.11    0.11    0.01    
Mean RMSE: 0.43969462099694
Mean MAE: 0.20698660945567227


We unpivot the group_profiles DataFrame to convert it into the long format, where each row represents a user-profile combination.

We create a Reader object specifying the rating scale used in the dataset (0 to 1).

We load the data into a Surprise Dataset object.

We initialize the SVD algorithm.

We perform 5-fold cross-validation using the SVD algorithm and evaluate the model's performance in terms of RMSE (Root Mean Squared Error) and MAE (Mean Absolute Error).

Finally, we print the mean RMSE and MAE across all folds.

In [16]:
from surprise.model_selection import train_test_split
from surprise import accuracy

# Split the dataset into training and test sets (80% train, 20% test)
trainset, testset = train_test_split(data, test_size=0.2)

# Initialize the SVD algorithm
algo = SVD()

# Train the algorithm on the training set
algo.fit(trainset)

# Predict ratings for the test set
predictions = algo.test(testset)

# Evaluate the predictions using RMSE and MAE
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

# Perform 5-fold cross-validation
results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

# Print the evaluation results
print("Train-test split RMSE:", rmse)
print("Train-test split MAE:", mae)
print("Mean RMSE (5-fold CV):", results['test_rmse'].mean())
print("Mean MAE (5-fold CV):", results['test_mae'].mean())

RMSE: 0.4344
MAE:  0.2074
Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.4376  0.4440  0.4336  0.4479  0.4459  0.4418  0.0054  
MAE (testset)     0.2081  0.2095  0.2060  0.2148  0.2147  0.2106  0.0035  
Fit time          1.09    1.11    1.08    1.08    1.09    1.09    0.01    
Test time         0.09    0.06    0.09    0.13    0.12    0.10    0.02    
Train-test split RMSE: 0.4344005501901025
Train-test split MAE: 0.20738484221974612
Mean RMSE (5-fold CV): 0.4417900530762367
Mean MAE (5-fold CV): 0.2106220728069923


Self Rejected Code Chunks

In [31]:
# Assuming 'data' is the DataFrame containing your dataset
# Extracting specific columns and saving to 'originaldata'
originaldata = data[['MovieID', 'Genres', 'UserID', 'Rating']].copy()

# Display the 'originaldata' DataFrame
print(originaldata.head())  # Display the first few rows to verify the extraction

   MovieID                       Genres  UserID  Rating
0        1  Animation|Children's|Comedy       1       5
1        1  Animation|Children's|Comedy       6       4
2        1  Animation|Children's|Comedy       8       4
3        1  Animation|Children's|Comedy       9       5
4        1  Animation|Children's|Comedy      10       5


In [32]:
# Split the 'Genres' column and create binary columns for each genre
genres = originaldata['Genres'].str.get_dummies(sep='|')

# Concatenate the binary genre columns with the original DataFrame
originaldata = pd.concat([originaldata, genres], axis=1)

# Drop the original 'Genres' column
originaldata.drop(columns=['Genres'], inplace=True)

# Display the updated DataFrame
originaldata.head()  # Display the first few rows to verify the changes

,MovieID,UserID,Rating,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,1,5,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,6,4,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,8,4,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,9,5,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,10,5,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0


Whether to normalize the ratings between -1 and 1 or 0 and 1 depends on the specific requirements of your dataset and the characteristics of the ratings. Normalization is not strictly required for KNN, but it can sometimes improve the performance of the algorithm, especially if the features are on different scales or have different units.

Here's a comparison between normalizing ratings between -1 and 1 and 0 and 1:

Normalizing between -1 and 1:

This approach scales the ratings to a range between -1 and 1.
Negative ratings may be preserved if they exist in the dataset.
This can be useful if your ratings can span both positive and negative values.
Normalizing between 0 and 1:

This approach scales the ratings to a range between 0 and 1.
All ratings are non-negative and fall within the same range.
This is commonly used when ratings are assumed to be non-negative or when the dataset does not contain negative ratings.
Choosing between these options depends on your specific use case and the nature of the ratings in your dataset. If your ratings are strictly non-negative, normalizing between 0 and 1 may be more appropriate. However, if your ratings can include both positive and negative values, normalizing between -1 and 1 may be preferable.

Normalizing ratings before using KNN can help ensure that all features contribute equally to the distance calculation. Features on different scales may otherwise have disproportionate influences on the distance metric, potentially biasing the results. However, KNN is not inherently sensitive to the scale of the features, so normalization is not always necessary.

To normalize ratings between 0 and 1, you can use Min-Max scaling, which linearly transforms the original ratings to the range [0, 1]. Here's how you can achieve this normalization using pandas:

In this code:

We first calculate the minimum (min_rating) and maximum (max_rating) ratings in the dataset.
Then, we use Min-Max scaling to normalize the ratings to the range [0, 1] using the formula:

normalized_rating = (rating - min_rating) / (max_rating - min_rating)

In [34]:
# Assuming 'Rating' is the column containing ratings

# Calculate the minimum and maximum ratings
min_rating = originaldata['Rating'].min()
max_rating = originaldata['Rating'].max()

# Normalize the ratings between 0 and 1
originaldata['Rating'] = (originaldata['Rating'] - min_rating) / (max_rating - min_rating)

# Display the updated DataFrame with normalized ratings
originaldata.head()

,MovieID,UserID,Rating,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,1,1.00,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,6,0.75,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,8,0.75,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,9,1.00,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,10,1.00,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [35]:
# Assuming 'originaldata' is the DataFrame
column_names = originaldata.columns.tolist()

# Print the column names
print("Column Names:", column_names)

Column Names: ['MovieID', 'UserID', 'Rating', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [6]:
# Define features for group formation
X = data[['MovieID', 'Genres', 'Director', 'Starring', 'UserID', 'Rating']]

# Initialize KNN model
k = 5  # Number of neighbors
knn = NearestNeighbors(n_neighbors=k, metric='gower_distance')

# Split the dataset into chunks (e.g., 10 chunks)
num_chunks = 10
chunk_size = len(data) // num_chunks

for i in range(num_chunks):
    # Select a chunk of data
    chunk = data.iloc[i * chunk_size: (i+1) * chunk_size]

    # Fit KNN model on the chunk
    knn.fit(chunk)

    # Perform group formation for the chunk
    # You can form groups similarly to previous examples, by finding nearest neighbors for each user and combining them into groups

    # Output the formed groups for the current chunk
    print(f"Groups formed for chunk {i+1}:")
    # print(groups)


InvalidParameterError: The 'metric' parameter of NearestNeighbors must be a str among {'canberra', 'p', 'jaccard', 'cityblock', 'precomputed', 'l2', 'hamming', 'haversine', 'pyfunc', 'seuclidean', 'sokalsneath', 'correlation', 'russellrao', 'l1', 'sqeuclidean', 'sokalmichener', 'yule', 'infinity', 'minkowski', 'dice', 'braycurtis', 'nan_euclidean', 'manhattan', 'euclidean', 'rogerstanimoto', 'mahalanobis', 'cosine', 'chebyshev'} or a callable. Got 'gower_distance' instead.

In [5]:
# Display the first few rows and the structure of the data
data.head(), data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 10 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   MovieID         1000209 non-null  int64 
 1   Title           1000209 non-null  object
 2   Genres          1000209 non-null  object
 3   Director        818669 non-null   object
 4   Starring        749995 non-null   object
 5   DBpediaURI      948960 non-null   object
 6   UserID          1000209 non-null  int64 
 7   Rating          1000209 non-null  int64 
 8   Music Composer  757744 non-null   object
 9   DBpedia URI     948960 non-null   object
dtypes: int64(3), object(7)
memory usage: 76.3+ MB


(   MovieID             Title                       Genres Director Starring  \
 0        1  Toy Story (1995)  Animation|Children's|Comedy      NaN      NaN   
 1        1  Toy Story (1995)  Animation|Children's|Comedy      NaN      NaN   
 2        1  Toy Story (1995)  Animation|Children's|Comedy      NaN      NaN   
 3        1  Toy Story (1995)  Animation|Children's|Comedy      NaN      NaN   
 4        1  Toy Story (1995)  Animation|Children's|Comedy      NaN      NaN   
 
   DBpediaURI  UserID  Rating Music Composer DBpedia URI  
 0        NaN       1       5            NaN         NaN  
 1        NaN       6       4            NaN         NaN  
 2        NaN       8       4            NaN         NaN  
 3        NaN       9       5            NaN         NaN  
 4        NaN      10       5            NaN         NaN  ,
 None)

In [6]:
# Extract the relevant columns
relevant_data = data[['MovieID', 'UserID', 'Rating']]

# Check for missing values in the relevant columns
missing_values = relevant_data.isnull().sum()

# Create a user-movie matrix
user_movie_matrix = relevant_data.pivot_table(index='UserID', columns='MovieID', values='Rating')

# Display the missing values count and a preview of the user-movie matrix
missing_values, user_movie_matrix.head()


(MovieID    0
 UserID     0
 Rating     0
 dtype: int64,
 MovieID  1     2     3     4     5     6     7     8     9     10    ...  \
 UserID                                                               ...   
 1         5.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
 2         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
 3         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
 4         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
 5         NaN   NaN   NaN   NaN   NaN   2.0   NaN   NaN   NaN   NaN  ...   
 
 MovieID  3943  3944  3945  3946  3947  3948  3949  3950  3951  3952  
 UserID                                                               
 1         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
 2         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
 3         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
 4         NaN   NaN   NaN   NaN   NaN   NaN   